## 1. Подключение библиотек

В этой ячейке мы загружаем все необходимые NuGet-пакеты для анализа данных, визуализации и вычисления корреляции.

**Используемые пакеты:**
- `Microsoft.Data.Analysis` — работа с табличными данными (аналог Pandas).
- `Microsoft.ML` — базовые инструменты ML.NET.
- `Plotly.NET.Interactive` и `Plotly.NET.CSharp` — построение интерактивных графиков прямо в ноутбуке.
- `Microsoft.DotNet.Interactive.Formatting` — улучшенное форматирование вывода.
- `MathNet.Numerics` — математические функции, в частности расчёт корреляции Пирсона.

In [1]:
// Основные пакеты для анализа данных
#r "nuget: Microsoft.Data.Analysis, 0.22.1"
#r "nuget: Microsoft.ML, 3.0.1"

// Пакеты для визуализации
#r "nuget: Plotly.NET.Interactive, 5.0.0"
#r "nuget: Plotly.NET.CSharp, 0.12.0"
#r "nuget: Microsoft.DotNet.Interactive.Formatting, 1.0.0-beta.25070.1"

// Пакет для расчёта корреляции
#r "nuget: MathNet.Numerics, 5.0.0"

The below script needs to be able to find the current output cell; this is an easy method to get it.

Installed Packages MathNet.Numerics, 5.0.0 Microsoft.Data.Analysis, 0.22.1 Microsoft.DotNet.Interactive.Formatting, 1.0.0-beta.25070.1 Microsoft.ML, 3.0.1 Plotly.NET.CSharp, 0.12.0 Plotly.NET.Interactive, 5.0.0

Loading extensions from `C:\Users\0\.nuget\packages\plotly.net.interactive\5.0.0\lib\netstandard2.1\Plotly.NET.Interactive.dll`

Loading extensions from `C:\Users\0\.nuget\packages\microsoft.data.analysis\0.22.1\interactive-extensions\dotnet\Microsoft.Data.Analysis.Interactive.dll`

## 2. Импорт пространств имён

Подключаем необходимые пространства имён, чтобы обращаться к классам и методам без указания полных путей.

**Основные импорты:**
- `Microsoft.Data.Analysis` — класс `DataFrame`.
- `Microsoft.ML` — контекст ML.NET.
- `Plotly.NET` и `Plotly.NET.CSharp` — построение графиков.
- `MathNet.Numerics.Statistics` — метод `Correlation.Pearson`.
- `System.Globalization` — культура для корректного парсинга чисел с точкой.
- `System.IO`, `System.Linq` — вспомогательные утилиты.

In [3]:
// Подключаем пространства имён
using Plotly.NET;
using PlotlyCS = Plotly.NET.CSharp.Chart;
using Microsoft.Data.Analysis;
using Microsoft.ML;
using Plotly.NET.CSharp;
using MathNet.Numerics.Statistics;
using Microsoft.DotNet.Interactive.Formatting;
using System.Linq;
using System.IO;
using System.Globalization;

## 3. Загрузка набора данных Iris

Загружаем CSV-файл `iris.csv`, расположенный в папке `Data` относительно текущего каталога ноутбука.

**Особенности:**
- Явно указываем имена столбцов и их типы (4 числовых признака — `float`, 1 категориальный — `string`).
- Используем `CultureInfo.InvariantCulture`, чтобы десятичная точка интерпретировалась корректно независимо от региональных настроек системы.

Результат — объект `DataFrame` с именем `df`.

In [4]:
var dataPath = Path.Combine(Directory.GetCurrentDirectory(), "Data", "iris.csv");

string[] columnNames = new[] { "SepalLength", "SepalWidth", "PetalLength", "PetalWidth", "Species" };
Type[] dataTypes = new[] { typeof(float), typeof(float), typeof(float), typeof(float), typeof(string) };

var df = DataFrame.LoadCsv(dataPath,
    separator: ',',
    header: true,
    columnNames: columnNames,
    dataTypes: dataTypes,
    cultureInfo: CultureInfo.InvariantCulture);

## 4. Основные характеристики датасета

Выводим сводную описательную статистику для числовых столбцов с помощью метода `Description()`.

**Отображаемые показатели:**
- Количество значений
- Среднее арифметическое
- Стандартное отклонение
- Минимум и максимум
- Квартили (25%, 50%, 75%)

Эти данные позволяют быстро оценить разброс и центральные тенденции каждого признака.

In [5]:
df.Head(5).Display();

index,SepalLength,SepalWidth,PetalLength,PetalWidth,Species
0,4.9,3,1.4,0.2,Iris-setosa
1,4.7,3.2,1.3,0.2,Iris-setosa
2,4.6,3.1,1.5,0.2,Iris-setosa
3,5,3.6,1.4,0.2,Iris-setosa
4,5.4,3.9,1.7,0.4,Iris-setosa


In [6]:
var desc = df.Description();
desc.Display();

index,Description,SepalLength,SepalWidth,PetalLength,PetalWidth
0,Length (excluding null values),149,149,149,149
1,Max,7.9,4.4,6.9,2.5
2,Min,4.3,2,1,0.1
3,Mean,5.848324,3.0510063,3.7744968,1.2053692


## 5. Визуализация распределения признака

Строим гистограмму для столбца `SepalLength` (длина чашелистика). Гистограмма показывает, как часто встречаются различные значения признака, и помогает оценить форму распределения (например, наличие нескольких пиков или асимметрию).

Мы используем `Plotly.NET.CSharp.Chart.Histogram`, а для устранения конфликта имён явно вызываем метод `Show`.

In [7]:
var sepalLength = df["SepalLength"].Cast<float>().ToArray();

var histogram = PlotlyCS.Histogram<float, float, string>(X: sepalLength)
    .WithTitle("Гистограмма длины чашелистика");

display(histogram);

<!-- Plotly chart will be drawn inside this DIV -->

## 6. Диаграмма рассеяния (Scatter Plot)

На этом графике мы отображаем зависимость между длиной лепестка (`PetalLength`) и шириной лепестка (`PetalWidth`).

Диаграмма рассеяния позволяет визуально оценить:
- Наличие корреляции между признаками.
- Структуру данных (например, разделение на кластеры, соответствующие разным видам ирисов).
- Возможные выбросы.

Каждая точка соответствует одному цветку.

In [ ]:
var petalLength = df["PetalLength"].Cast<float>().ToArray();
var petalWidth = df["PetalWidth"].Cast<float>().ToArray();

var chart = PlotlyCS.Scatter<float, float, string>(
    x: petalLength,
    y: petalWidth,
    mode: StyleParam.Mode.Markers)
    .WithTitle("Диаграмма рассеяния: Длина и ширина лепестка");

display(chart);

<!-- Plotly chart will be drawn inside this DIV -->

## 7. Корреляция признаков

Вычисляем коэффициенты корреляции Пирсона между всеми парами числовых признаков датасета.

**Интерпретация:**
- Значения близкие к **+1** указывают на сильную положительную связь (при увеличении одного признака другой также растёт).
- Значения близкие к **-1** — сильная отрицательная связь.
- Значения около **0** — линейная связь отсутствует.

Результаты выводятся в консоль в виде удобной таблицы.

Особое внимание стоит обратить на высокую корреляцию между длиной и шириной лепестка (ожидается > 0.95).

In [ ]:
// Извлекаем числовые столбцы и преобразуем в double[]
var sepalLength = df["SepalLength"].Cast<float>().Select(x => (double)x).ToArray();
var sepalWidth  = df["SepalWidth"].Cast<float>().Select(x => (double)x).ToArray();
var petalLength = df["PetalLength"].Cast<float>().Select(x => (double)x).ToArray();
var petalWidth  = df["PetalWidth"].Cast<float>().Select(x => (double)x).ToArray();

// Расчёт корреляций между всеми парами признаков
var corr_SL_SW = Correlation.Pearson(sepalLength, sepalWidth);
var corr_SL_PL = Correlation.Pearson(sepalLength, petalLength);
var corr_SL_PW = Correlation.Pearson(sepalLength, petalWidth);

var corr_SW_PL = Correlation.Pearson(sepalWidth, petalLength);
var corr_SW_PW = Correlation.Pearson(sepalWidth, petalWidth);

var corr_PL_PW = Correlation.Pearson(petalLength, petalWidth);

// Вывод в виде симпатичной таблицы
Console.WriteLine("Матрица корреляции Пирсона:");
Console.WriteLine($"SepalLength - SepalWidth : {corr_SL_SW:F4}");
Console.WriteLine($"SepalLength - PetalLength: {corr_SL_PL:F4}");
Console.WriteLine($"SepalLength - PetalWidth : {corr_SL_PW:F4}");
Console.WriteLine($"SepalWidth  - PetalLength: {corr_SW_PL:F4}");
Console.WriteLine($"SepalWidth  - PetalWidth : {corr_SW_PW:F4}");
Console.WriteLine($"PetalLength - PetalWidth : {corr_PL_PW:F4}");

Матрица корреляции Пирсона:
SepalLength - SepalWidth : -0,1038
SepalLength - PetalLength: 0,8713
SepalLength - PetalWidth : 0,8170
SepalWidth  - PetalLength: -0,4152
SepalWidth  - PetalWidth : -0,3507
PetalLength - PetalWidth : 0,9623
